# 1D Kalman Filter：位置和速度估计

这个 notebook 用一维匀速小车示例理解 Kalman Filter 的基本流程：

1. 生成真实轨迹 `ground truth`
2. 生成带噪声的位置测量 `measurement`
3. 创建一维匀速模型的 `KalmanFilter`
4. 循环执行 `predict()` 和 `update()`
5. 画出真实位置、测量值和滤波估计结果

## 1. 导入依赖和设置路径

`PROJECT_ROOT` 指向项目根目录，后面保存图片会固定保存到 `figures/kf_1d_result.png`。

In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_ROOT

WindowsPath('e:/申博材料/机器人相关学习/阶段一_基础补齐/01_FilterPy状态估计_移动机器人定位')

## 2. 设置参数

- `true_velocity`：真实世界里的速度
- `initial_velocity_guess`：滤波器一开始猜的速度
- `measurement_noise_std`：测量噪声标准差
- `process_noise_var`：过程噪声方差，用来构造 `Q`

In [2]:
dt = 1.0
num_steps = 80
initial_position = 0.0
true_velocity = 1.0
initial_velocity_guess = 0.0
measurement_noise_std = 4.0
process_noise_var = 0.01
random_seed = 42

rng = np.random.default_rng(random_seed)

## 3. 生成真实轨迹

这里不调用 FilterPy，只用一维匀速公式：

`position = initial_position + true_velocity * time`

In [ ]:
def simulate_ground_truth(num_steps, dt, initial_position, true_velocity):
    times = np.arange(num_steps) * dt
    true_positions = initial_position + true_velocity * times
    return times, true_positions


times, true_positions = simulate_ground_truth(
    num_steps=num_steps,
    dt=dt,
    initial_position=initial_position,
    true_velocity=true_velocity,
)

times[:5], true_positions[:5]

## 4. 生成带噪声测量

传感器测量值用下面的关系模拟：

`measurement = true_position + Gaussian noise`

In [ ]:
def simulate_noisy_measurements(true_positions, measurement_noise_std, rng):
    noise = rng.normal(
        loc=0.0,
        scale=measurement_noise_std,
        size=true_positions.shape,
    )
    measurements = true_positions + noise
    return measurements


measurements = simulate_noisy_measurements(
    true_positions=true_positions,
    measurement_noise_std=measurement_noise_std,
    rng=rng,
)

measurements[:5]

## 5. 创建 Kalman Filter

状态变量：

`x = [position, velocity]`

测量变量：

`z = [position]`

关键矩阵：

- `F = [[1, dt], [0, 1]]`：一维匀速运动模型
- `H = [[1, 0]]`：传感器只测位置，不测速度
- `P`：初始状态不确定性
- `R`：测量噪声方差
- `Q`：过程噪声矩阵

In [ ]:
def create_kalman_filter(dt, initial_position, initial_velocity,
                         measurement_noise_std, process_noise_var):
    kf = KalmanFilter(dim_x=2, dim_z=1)

    kf.x = np.array([initial_position, initial_velocity])
    kf.F = np.array([
        [1.0, dt],
        [0.0, 1.0],
    ])
    kf.H = np.array([[1.0, 0.0]])
    kf.P = np.eye(2) * 500.0
    kf.R = np.array([[measurement_noise_std ** 2]])
    kf.Q = Q_discrete_white_noise(
        dim=2,
        dt=dt,
        var=process_noise_var,
    )

    return kf


kf = create_kalman_filter(
    dt=dt,
    initial_position=initial_position,
    initial_velocity=initial_velocity_guess,
    measurement_noise_std=measurement_noise_std,
    process_noise_var=process_noise_var,
)

print("x shape:", kf.x.shape)
print("F shape:", kf.F.shape)
print("H shape:", kf.H.shape)
print("P shape:", kf.P.shape)
print("R shape:", kf.R.shape)
print("Q shape:", kf.Q.shape)

## 6. 运行 predict/update 循环

每一个测量值 `z` 进来后：

1. `kf.predict()`：根据 `F` 和 `Q` 做预测
2. `kf.update(np.array([z]))`：根据测量值、`H` 和 `R` 修正预测
3. 保存当前 `kf.x`

In [ ]:
def run_kalman_filter(measurements, dt, initial_position, initial_velocity,
                      measurement_noise_std, process_noise_var):
    kf = create_kalman_filter(
        dt=dt,
        initial_position=initial_position,
        initial_velocity=initial_velocity,
        measurement_noise_std=measurement_noise_std,
        process_noise_var=process_noise_var,
    )

    estimates = np.zeros((len(measurements), 2))

    for i, z in enumerate(measurements):
        kf.predict()
        kf.update(np.array([z]))

        estimates[i, 0] = kf.x[0]
        estimates[i, 1] = kf.x[1]

    return estimates


estimates = run_kalman_filter(
    measurements=measurements,
    dt=dt,
    initial_position=initial_position,
    initial_velocity=initial_velocity_guess,
    measurement_noise_std=measurement_noise_std,
    process_noise_var=process_noise_var,
)

estimates[:5]

## 7. 画图并保存

图片会保存到：

`figures/kf_1d_result.png`

In [ ]:
def plot_results(times, true_positions, measurements, estimates, output_path):
    plt.figure(figsize=(10, 5))

    plt.plot(times, true_positions, label="Ground truth", linewidth=2)
    plt.scatter(times, measurements, s=18, alpha=0.5, label="Noisy measurement")
    plt.plot(times, estimates[:, 0], label="Kalman estimate", linewidth=2)

    plt.xlabel("Time [s]")
    plt.ylabel("Position")
    plt.title("1D Kalman Filter: Position Estimation")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150)
    plt.show()


output_path = PROJECT_ROOT / "figures" / "kf_1d_result.png"
plot_results(
    times=times,
    true_positions=true_positions,
    measurements=measurements,
    estimates=estimates,
    output_path=output_path,
)

output_path

## 8. 可选：观察速度估计

测量值里没有速度，但 Kalman Filter 会通过连续的位置测量逐步估计速度。

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(times, estimates[:, 1], label="Estimated velocity", linewidth=2)
plt.axhline(true_velocity, color="black", linestyle="--", label="True velocity")
plt.xlabel("Time [s]")
plt.ylabel("Velocity")
plt.title("1D Kalman Filter: Velocity Estimation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()